<a href="https://colab.research.google.com/github/dooniteaadi/compiler-design/blob/main/compiler_labfile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.lexical Analysis

In [ ]:
import re
import keyword

def analyze_lexical_units(source_code):
    # Using structured pattern matching and updated naming to identify lexemes
    lex_pattern = r'\b\d+\b|[a-zA-Z_]\w*|==|!=|<=|>=|[+\-*/%=<>]|[(){};,]'
    lexemes = re.findall(lex_pattern, source_code)

    print(f"{'TOKEN':<15}{'TYPE'}")
    print("----------------------------")

    operators = {"+", "-", "*", "/", "%", "=", "<", ">", "==", "!=", "<=", ">="}

    for lex in lexemes:
        if keyword.iskeyword(lex):
            category = "Keyword"
        elif lex.isidentifier():
            category = "Identifier"
        elif lex.isdigit():
            category = "Constant"
        elif lex in operators:
            category = "Operator"
        else:
            category = "Special Symbol"

        print(f"{lex:<15} {category}")

input_code = """
int a = 10;
int b = 20;
if (a < b) {
    a = a + b;
}
"""

analyze_lexical_units(input_code)
print("\nADITYA SILSWAL 2472125")

TOKEN          TYPE
----------------------------
int             Identifier
a               Identifier
=               Operator
10              Constant
;               Special Symbol
int             Identifier
b               Identifier
=               Operator
20              Constant
;               Special Symbol
if              Keyword
(               Special Symbol
a               Identifier
<               Operator
b               Identifier
)               Special Symbol
{               Special Symbol
a               Identifier
=               Operator
a               Identifier
+               Operator
b               Identifier
;               Special Symbol
}               Special Symbol

ADITYA SILSWAL 247125


2.First() and Follow()

In [ ]:
cfg_rules = {
    'E': ['TR'],
    'R': ['+TR', '#'],
    'T': ['FY'],
    'Y': ['*FY', '#'],
    'F': ['(E)', 'i']
}

first_sets = {non_term: set() for non_term in cfg_rules}
follow_sets = {non_term: set() for non_term in cfg_rules}

def compute_first(sym):
    if sym not in cfg_rules:
        return {sym}

    if first_sets[sym]:
        return first_sets[sym]

    for prod in cfg_rules[sym]:
        if prod == '#':
            first_sets[sym].add('#')
            continue

        for char in prod:
            char_first = compute_first(char)
            first_sets[sym].update(char_first - {'#'})

            if '#' not in char_first:
                break
        else:
            first_sets[sym].add('#')

    return first_sets[sym]

def compute_follow():
    follow_sets['E'].add('$')
    has_changed = True

    while has_changed:
        has_changed = False

        for nt in cfg_rules:
            for prod in cfg_rules[nt]:
                for idx, symbol in enumerate(prod):
                    if symbol in cfg_rules:
                        prev_size = len(follow_sets[symbol])
                        suffix = prod[idx + 1:]

                        if suffix:
                            suffix_first = set()
                            can_be_null = True

                            for s in suffix:
                                res = compute_first(s)
                                suffix_first.update(res - {'#'})
                                if '#' not in res:
                                    can_be_null = False
                                    break

                            follow_sets[symbol].update(suffix_first)
                            if can_be_null:
                                follow_sets[symbol].update(follow_sets[nt])
                        else:
                            follow_sets[symbol].update(follow_sets[nt])

                        if len(follow_sets[symbol]) > prev_size:
                            has_changed = True

for nt in cfg_rules:
    compute_first(nt)

compute_follow()

print("FIRST SETS:")
for nt in sorted(cfg_rules.keys(), key=lambda x: ['E', 'R', 'T', 'Y', 'F'].index(x)):
    print(f"FIRST( {nt} ) = {first_sets[nt]}")

print("\nFOLLOW SETS:")
for nt in sorted(cfg_rules.keys(), key=lambda x: ['E', 'R', 'T', 'Y', 'F'].index(x)):
    print(f"FOLLOW( {nt} ) = {follow_sets[nt]}")

print("\nADITYA SILSWAL 2472125")

FIRST SETS:
FIRST( E ) = {'(', 'i'}
FIRST( R ) = {'+', '#'}
FIRST( T ) = {'(', 'i'}
FIRST( Y ) = {'*', '#'}
FIRST( F ) = {'(', 'i'}

FOLLOW SETS:
FOLLOW( E ) = {')', '$'}
FOLLOW( R ) = {')', '$'}
FOLLOW( T ) = {'+', ')', '$'}
FOLLOW( Y ) = {'+', ')', '$'}
FOLLOW( F ) = {'+', ')', '$', '*'}

ADITYA SILSWAL 247125


LL(1) parser

In [ ]:
# LL(1) Parser Implementation with streamlined stack operations and structured print

ll1_table = {
    'E': {'i': 'TQ', '(': 'TQ'},
    'Q': {'+': '+TQ', ')': '#', '$': '#'},
    'T': {'i': 'FR', '(': 'FR'},
    'R': {'+': '#', '*': '*FR', ')': '#', '$': '#'},
    'F': {'i': 'i', '(': '(E)'}
}

def parse_ll1(expr):
    parse_stack = ['$', 'E']
    input_buffer = expr + '$'
    cursor = 0

    print("\nSTACK\t\tINPUT\t\tACTION")
    print("-----------------------------------------------")

    while len(parse_stack) > 0:
        top_symbol = parse_stack[-1]
        current_input = input_buffer[cursor:]
        current_char = input_buffer[cursor]

        print(f"{''.join(parse_stack):<15} {current_input:<15} ", end='')

        if top_symbol == current_char == '$':
            print("ACCEPTED")
            print("\nADITYA SILSWAL 2472125")
            return

        if top_symbol == current_char:
            parse_stack.pop()
            cursor += 1
            print(f"Match {current_char}")
        elif top_symbol in ll1_table:
            production_rule = ll1_table[top_symbol].get(current_char)

            if production_rule is None:
                print("ERROR: String Rejected")
                print("\nADITYA SILSWAL 2472125")
                return

            parse_stack.pop()
            if production_rule != '#':
                for symbol in reversed(production_rule):
                    parse_stack.append(symbol)
            print(f"{top_symbol} -> {production_rule}")
        else:
            print("ERROR: String Rejected")
            print("\nADITYA SILSWAL 2472125")
            return

    print("REJECTED")
    print("\nADITYA SILSWAL 2472125")

user_input = input("Enter input string (use i for identifier): ")
parse_ll1(user_input)

Recursive Descent parser

In [ ]:
# Recursive Descent Parser using a class wrapper to avoid global keyword overuse

class RecursiveDescentParser:
    def __init__(self, text):
        self.text = text
        self.pos = 0

    def match_char(self, target):
        if self.pos < len(self.text) and self.text[self.pos] == target:
            self.pos += 1
            return True
        return False

    # E -> TQ
    def parse_E(self):
        return self.parse_T() and self.parse_Q()

    # Q -> +TQ | epsilon
    def parse_Q(self):
        if self.match_char('+'):
            return self.parse_T() and self.parse_Q()
        return True

    # T -> FR
    def parse_T(self):
        return self.parse_F() and self.parse_R()

    # R -> *FR | epsilon
    def parse_R(self):
        if self.match_char('*'):
            return self.parse_F() and self.parse_R()
        return True

    # F -> (E) | i
    def parse_F(self):
        if self.match_char('i'):
            return True
        elif self.match_char('('):
            if self.parse_E():
                return self.match_char(')')
        return False

    def run_parser(self):
        if self.parse_E() and self.pos == len(self.text):
            print("String Accepted")
        else:
            print("String Rejected")
        print("\nADITYA SILSWAL 2472125")

input_data = input("Enter the input string (use i for identifier): ")
parser_instance = RecursiveDescentParser(input_data)
parser_instance.run_parser()

LR(0) PARSER

In [ ]:
# Program to Check LR(0) Grammar - rewritten using clean comprehensions and list utilities

grammar_spec = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

rule_list = [("S'", "S")]
for lhs_var, rhs_options in grammar_spec.items():
    for option in rhs_options:
        rule_list.append((lhs_var, option))

nt_symbols = set(grammar_spec.keys()) | {"S'"}
t_symbols = {char for _, rhs in rule_list for char in rhs if char not in nt_symbols} | {'$'}

def compute_closure(items):
    closure_set = set(items)
    active = True
    while active:
        active = False
        for p_idx, dot_pos in list(closure_set):
            lhs, rhs = rule_list[p_idx]
            if dot_pos < len(rhs):
                next_sym = rhs[dot_pos]
                if next_sym in nt_symbols:
                    for idx, (l, r) in enumerate(rule_list):
                        if l == next_sym and (idx, 0) not in closure_set:
                            closure_set.add((idx, 0))
                            active = True
    return frozenset(closure_set)

def compute_goto(items, symbol):
    moved_items = {
        (p, dot + 1) for p, dot in items
        if dot < len(rule_list[p][1]) and rule_list[p][1][dot] == symbol
    }
    return compute_closure(moved_items) if moved_items else frozenset()

lr0_states = [compute_closure({(0, 0)})]
state_transitions = {}

state_idx = 0
while state_idx < len(lr0_states):
    for sym in sorted(t_symbols | nt_symbols - {'$', "S'"}):
        target_state = compute_goto(lr0_states[state_idx], sym)
        if target_state:
            if target_state not in lr0_states:
                lr0_states.append(target_state)
            state_transitions[(state_idx, sym)] = lr0_states.index(target_state)
    state_idx += 1

print("LR(0) ITEM SETS")
for idx, st in enumerate(lr0_states):
    print(f"\nI {idx}")
    for p, dot in sorted(st):
        lhs, rhs = rule_list[p]
        dotted_rhs = rhs[:dot] + "." + rhs[dot:]
        print(f"{lhs} -> {dotted_rhs}")

action_table = {}
goto_table = {}

for i in range(len(lr0_states)):
    for term in t_symbols:
        action_table[(i, term)] = ''
    for nt in grammar_spec:
        goto_table[(i, nt)] = ''

has_conflict = False

def register_action(key, act_val):
    global has_conflict
    if action_table[key] != '' and action_table[key] != act_val:
        has_conflict = True
    else:
        action_table[key] = act_val

for i, st in enumerate(lr0_states):
    for term in t_symbols - {'$'}:
        if (i, term) in state_transitions:
            register_action((i, term), f"S{state_transitions[(i, term)]}")

    for nt in grammar_spec:
        if (i, nt) in state_transitions:
            goto_table[(i, nt)] = state_transitions[(i, nt)]

    for p, dot in st:
        lhs, rhs = rule_list[p]
        if dot == len(rhs):
            if p == 0:
                register_action((i, '$'), 'ACCEPT')
            else:
                for term in t_symbols:
                    register_action((i, term), f"R{p}")

print("\nACTION TABLE")
for i in range(len(lr0_states)):
    print(f"State {i} : ", end="")
    for t in sorted(t_symbols):
        print(f"{t} = {action_table[(i, t)]}  ", end="")
    print()

print("\nGOTO TABLE")
for i in range(len(lr0_states)):
    print(f"State {i} : ", end="")
    for nt in grammar_spec:
        print(f"{nt} = {goto_table[(i, nt)]}  ", end="")
    print()

if has_conflict:
    print("\nGrammar is NOT LR(0)")
else:
    print("\nGrammar IS LR(0)")

print("\nADITYA SILSWAL 2472125")

SLR(1) PARSER

In [ ]:
# Program to Check SLR(1) Grammar - Optimized logic and updated loops

grammar_rules = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

rule_list = [("S'", "S")]
for lhs, options in grammar_rules.items():
    for opt in options:
        rule_list.append((lhs, opt))

nt_set = set(grammar_rules.keys()) | {"S'"}
t_set = {char for _, rhs in rule_list for char in rhs if char not in nt_set} | {'$'}

def find_first_set(symbol):
    if symbol not in grammar_rules:
        return {symbol}
    accumulated = set()
    for prod in grammar_rules[symbol]:
        if prod == '#':
            accumulated.add('#')
            continue
        is_nullable = True
        for char in prod:
            res = find_first_set(char)
            accumulated.update(res - {'#'})
            if '#' not in res:
                is_nullable = False
                break
        if is_nullable:
            accumulated.add('#')
    return accumulated

follow_sets = {nt: set() for nt in grammar_rules}
follow_sets['S'].add('$')

updating = True
while updating:
    updating = False
    for lhs, rhs in rule_list[1:]:
        for idx, char in enumerate(rhs):
            if char in grammar_rules:
                size_before = len(follow_sets[char])
                suffix = rhs[idx + 1:]
                if suffix:
                    nullable_trail = True
                    for sym in suffix:
                        temp = find_first_set(sym)
                        follow_sets[char].update(temp - {'#'})
                        if '#' not in temp:
                            nullable_trail = False
                            break
                    if nullable_trail:
                        follow_sets[char].update(follow_sets[lhs])
                else:
                    follow_sets[char].update(follow_sets[lhs])

                if len(follow_sets[char]) > size_before:
                    updating = True

def get_closure(items):
    closure_items = set(items)
    changing = True
    while changing:
        changing = False
        for p_idx, dot in list(closure_items):
            lhs, rhs = rule_list[p_idx]
            if dot < len(rhs):
                sym = rhs[dot]
                if sym in nt_set:
                    for idx, (l, r) in enumerate(rule_list):
                        if l == sym and (idx, 0) not in closure_items:
                            closure_items.add((idx, 0))
                            changing = True
    return frozenset(closure_items)

def compute_goto(items, symbol):
    stepped = {(p, dot + 1) for p, dot in items if dot < len(rule_list[p][1]) and rule_list[p][1][dot] == symbol}
    return get_closure(stepped) if stepped else frozenset()

states_list = [get_closure({(0, 0)})]
state_links = {}

idx = 0
while idx < len(states_list):
    for sym in sorted((t_set | nt_set) - {'$', "S'"}):
        next_state = compute_goto(states_list[idx], sym)
        if next_state:
            if next_state not in states_list:
                states_list.append(next_state)
            state_links[(idx, sym)] = states_list.index(next_state)
    idx += 1

actions = {}
gotos = {}
for i in range(len(states_list)):
    for t in t_set:
        actions[(i, t)] = ''
    for nt in grammar_rules:
        gotos[(i, nt)] = ''

s_conflict = False
def record_action(key, value):
    global s_conflict
    if actions[key] != '' and actions[key] != value:
        s_conflict = True
    else:
        actions[key] = value

for i, st in enumerate(states_list):
    for t in t_set - {'$'}:
        if (i, t) in state_links:
            record_action((i, t), f"S{state_links[(i, t)]}")
    for nt in grammar_rules:
        if (i, nt) in state_links:
            gotos[(i, nt)] = state_links[(i, nt)]
    for p, dot in st:
        lhs, rhs = rule_list[p]
        if dot == len(rhs):
            if p == 0:
                record_action((i, '$'), 'ACCEPT')
            else:
                for t in follow_sets[lhs]:
                    record_action((i, t), f"R{p}")

print("FOLLOW SETS")
for nt in grammar_rules:
    print(f"FOLLOW( {nt} ) = {follow_sets[nt]}")

print("\nLR(0) ITEM SETS")
for i, st in enumerate(states_list):
    print(f"\nI {i}")
    for p, dot in sorted(st):
        lhs, rhs = rule_list[p]
        item_repr = rhs[:dot] + "." + rhs[dot:]
        print(f"{lhs} -> {item_repr}")

print("\nACTION TABLE")
for i in range(len(states_list)):
    print(f"State {i} : ", end="")
    for t in sorted(t_set):
        print(f"{t} = {actions[(i, t)]}  ", end="")
    print()

print("\nGOTO TABLE")
for i in range(len(states_list)):
    print(f"State {i} : ", end="")
    for nt in grammar_rules:
        print(f"{nt} = {gotos[(i, nt)]}  ", end="")
    print()

if s_conflict:
    print("\nGrammar is NOT SLR(1)")
else:
    print("\nGrammar IS SLR(1)")

print("\nADITYA SILSWAL 2472125")

CLR PARSER

In [ ]:
# Program to Check CLR(1) Grammar - Rewritten using simplified loops and list structures

cfg_rules = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

rule_list = [("S'", "S")]
for lhs, options in cfg_rules.items():
    for opt in options:
        rule_list.append((lhs, opt))

nt_set = set(cfg_rules) | {"S'"}
t_set = {char for _, rhs in rule_list for char in rhs if char not in nt_set} | {'$'}

def compute_sequence_first(seq):
    accumulated = set()
    if not seq:
        return {'#'}
    for sym in seq:
        if sym not in nt_set:
            accumulated.add(sym)
            return accumulated
        is_nullable = False
        for prod in cfg_rules[sym]:
            if prod == '#':
                is_nullable = True
            else:
                accumulated.update(compute_sequence_first(prod) - {'#'})
                if '#' in compute_sequence_first(prod):
                    is_nullable = True
        if not is_nullable:
            return accumulated
    accumulated.add('#')
    return accumulated

def find_lr1_closure(items):
    closure_items = set(items)
    changing = True
    while changing:
        changing = False
        for p_idx, dot, la in list(closure_items):
            lhs, rhs = rule_list[p_idx]
            if dot < len(rhs):
                B = rhs[dot]
                if B in nt_set:
                    beta_lookahead = rhs[dot + 1:] + la
                    lookaheads = compute_sequence_first(beta_lookahead) - {'#'}
                    for idx, (l, r) in enumerate(rule_list):
                        if l == B:
                            for a in lookaheads:
                                entry = (idx, 0, a)
                                if entry not in closure_items:
                                    closure_items.add(entry)
                                    changing = True
    return frozenset(closure_items)

def compute_lr1_goto(items, symbol):
    shifted_items = {(p, dot + 1, la) for p, dot, la in items if dot < len(rule_list[p][1]) and rule_list[p][1][dot] == symbol}
    return find_lr1_closure(shifted_items) if shifted_items else frozenset()

clr_states = [find_lr1_closure({(0, 0, '$')})]
state_transitions = {}

idx = 0
while idx < len(clr_states):
    symbols = sorted((t_set | nt_set) - {'$'})
    for sym in symbols:
        next_st = compute_lr1_goto(clr_states[idx], sym)
        if next_st:
            if next_st not in clr_states:
                clr_states.append(next_st)
            state_transitions[(idx, sym)] = clr_states.index(next_st)
    idx += 1

actions = {}
gotos = {}
for i in range(len(clr_states)):
    for t in t_set:
        actions[(i, t)] = ''
    for nt in cfg_rules:
        gotos[(i, nt)] = ''

c_conflict = False
def record_action(key, val):
    global c_conflict
    if actions[key] != '' and actions[key] != val:
        c_conflict = True
    else:
        actions[key] = val

for i, st in enumerate(clr_states):
    for t in t_set - {'$'}:
        if (i, t) in state_transitions:
            record_action((i, t), f"S{state_transitions[(i, t)]}")
    for nt in cfg_rules:
        if (i, nt) in state_transitions:
            gotos[(i, nt)] = state_transitions[(i, nt)]
    for p, dot, la in st:
        lhs, rhs = rule_list[p]
        if dot == len(rhs):
            if p == 0:
                record_action((i, '$'), 'ACCEPT')
            else:
                record_action((i, la), f"R{p}")

print("CLR(1) ITEM SETS")
for i, st in enumerate(clr_states):
    print(f"\nI {i}")
    for p, dot, la in sorted(st):
        lhs, rhs = rule_list[p]
        dotted_rhs = rhs[:dot] + "." + rhs[dot:]
        print(f"{lhs} -> {dotted_rhs} , {la}")

print("\nACTION TABLE")
for i in range(len(clr_states)):
    print(f"State {i} : ", end="")
    for t in sorted(t_set):
        print(f"{t} = {actions[(i, t)]}  ", end="")
    print()

print("\nGOTO TABLE")
for i in range(len(clr_states)):
    print(f"State {i} : ", end="")
    for nt in cfg_rules:
        print(f"{nt} = {gotos[(i, nt)]}  ", end="")
    print()

if c_conflict:
    print("\nGrammar is NOT CLR(1)")
else:
    print("\nGrammar IS CLR(1)")

print("\nADITYA SILSWAL 2472125")

LALR PARSER

In [ ]:
# Program to Check LALR(1) Grammar - Streamlined state-merging logic

grammar_spec = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

rule_list = [("S'", "S")]
for lhs, options in grammar_spec.items():
    for opt in options:
        rule_list.append((lhs, opt))

nt_set = set(grammar_spec) | {"S'"}
t_set = {char for _, rhs in rule_list for char in rhs if char not in nt_set} | {'$'}

def evaluate_sequence_first(seq):
    accumulated = set()
    if not seq:
        return {'#'}
    for sym in seq:
        if sym not in nt_set:
            accumulated.add(sym)
            return accumulated
        is_nullable = False
        for prod in grammar_spec[sym]:
            if prod == '#':
                is_nullable = True
            else:
                res = evaluate_sequence_first(prod)
                accumulated.update(res - {'#'})
                if '#' in res:
                    is_nullable = True
        if not is_nullable:
            return accumulated
    accumulated.add('#')
    return accumulated

def get_lr1_closure(items):
    closure_items = set(items)
    changing = True
    while changing:
        changing = False
        for p_idx, dot, la in list(closure_items):
            lhs, rhs = rule_list[p_idx]
            if dot < len(rhs):
                B = rhs[dot]
                if B in nt_set:
                    beta_lookahead = rhs[dot + 1:] + la
                    lookaheads = evaluate_sequence_first(beta_lookahead) - {'#'}
                    for idx, (l, r) in enumerate(rule_list):
                        if l == B:
                            for a in lookaheads:
                                entry = (idx, 0, a)
                                if entry not in closure_items:
                                    closure_items.add(entry)
                                    changing = True
    return frozenset(closure_items)

def compute_lr1_goto(items, symbol):
    shifted_items = {(p, dot + 1, la) for p, dot, la in items if dot < len(rule_list[p][1]) and rule_list[p][1][dot] == symbol}
    return get_lr1_closure(shifted_items) if shifted_items else frozenset()

clr_states = [get_lr1_closure({(0, 0, '$')})]
clr_transitions = {}

idx = 0
while idx < len(clr_states):
    symbols = sorted((t_set | nt_set) - {'$'})
    for sym in symbols:
        next_st = compute_lr1_goto(clr_states[idx], sym)
        if next_st:
            if next_st not in clr_states:
                clr_states.append(next_st)
            clr_transitions[(idx, sym)] = clr_states.index(next_st)
    idx += 1

def get_lr0_core(state):
    return frozenset((p, dot) for p, dot, la in state)

merged_states = []
core_to_merged_idx = {}
old_to_new_mapping = {}

for i, state in enumerate(clr_states):
    core_items = get_lr0_core(state)
    if core_items not in core_to_merged_idx:
        core_to_merged_idx[core_items] = len(merged_states)
        merged_states.append(set())
    new_idx = core_to_merged_idx[core_items]
    old_to_new_mapping[i] = new_idx
    merged_states[new_idx].update(state)

merged_states = [frozenset(s) for s in merged_states]
merged_transitions = {}

for (i, symbol), j in clr_transitions.items():
    key = (old_to_new_mapping[i], symbol)
    val = old_to_new_mapping[j]
    merged_transitions[key] = val

actions = {}
gotos = {}
for i in range(len(merged_states)):
    for t in t_set:
        actions[(i, t)] = ''
    for nt in grammar_spec:
        gotos[(i, nt)] = ''

l_conflict = False
def record_action(key, value):
    global l_conflict
    if actions[key] != '' and actions[key] != value:
        l_conflict = True
    else:
        actions[key] = value

for i, st in enumerate(merged_states):
    for t in t_set - {'$'}:
        if (i, t) in merged_transitions:
            record_action((i, t), f"S{merged_transitions[(i, t)]}")
    for nt in grammar_spec:
        if (i, nt) in merged_transitions:
            gotos[(i, nt)] = merged_transitions[(i, nt)]
    for p, dot, la in st:
        lhs, rhs = rule_list[p]
        if dot == len(rhs):
            if p == 0:
                record_action((i, '$'), 'ACCEPT')
            else:
                record_action((i, la), f"R{p}")

print("LALR(1) ITEM SETS")
for i, st in enumerate(merged_states):
    print(f"\nI {i}")
    for p, dot, la in sorted(st):
        lhs, rhs = rule_list[p]
        dotted_rhs = rhs[:dot] + "." + rhs[dot:]
        print(f"{lhs} -> {dotted_rhs} , {la}")

print("\nACTION TABLE")
for i in range(len(merged_states)):
    print(f"State {i} : ", end="")
    for t in sorted(t_set):
        print(f"{t} = {actions[(i, t)]}  ", end="")
    print()

print("\nGOTO TABLE")
for i in range(len(merged_states)):
    print(f"State {i} : ", end="")
    for nt in grammar_spec:
        print(f"{nt} = {gotos[(i, nt)]}  ", end="")
    print()

if l_conflict:
    print("\nGrammar is NOT LALR(1)")
else:
    print("\nGrammar IS LALR(1)")

print("\nADITYA SILSWAL 2472125")